# Recency-weighted (decayed) RAPM

**Before running:** same kernel as the RAPM pipeline notebooks (needs `pandas`, `numpy`, `scipy`, `scikit-learn`).

Your existing multi-year RAPM (`rapm_3yr_2025_26.csv`, `rapm_5yr_2025_26.csv`) equal-pools possessions across seasons -- a possession from 2021-22 counts exactly as much as one from 2025-26. That's a real design choice with a real cost: it treats a player's game from 4 years ago as equally informative about who they are *today* as a game from last week, which isn't true for anyone whose role, health, or team situation has changed.

This builds the alternative: every possession still counts, but older seasons count for less, via an exponential decay on the sample weight. No hard cutoff is needed the way "3-year" or "5-year" implies one -- decay naturally fades old seasons toward irrelevance rather than truncating them, so this uses the full available history (2015-16 through 2025-26, 11 seasons, no gaps since `rapm_possessions2018.csv` was consolidated earlier this project).

This reuses your existing `rapm_pipeline` package (imported from `PythonBasketball/play_by_play/`) rather than re-implementing the RAPM math -- `rapm.run()` gained one new optional parameter (`extra_weight_col`) to support this, which is a pure addition: every existing call site (the season pipeline, the 3yr/5yr windows) is unaffected since it defaults to `None` and reproduces the exact original behavior.

In [ ]:
import sys, os
import numpy as np
import pandas as pd

RAPM_PIPELINE_DIR = os.path.join(os.path.dirname(os.getcwd()), "PythonBasketball", "play_by_play")
sys.path.insert(0, RAPM_PIPELINE_DIR)

from rapm_pipeline import rapm, nba_client, config

HERE = os.getcwd()  # .../PlayerValueMetrics
OUT_DIR = os.path.join(HERE, "data")
POSS_DIR = os.path.join(RAPM_PIPELINE_DIR, "data")

print("Reading possession files from:", POSS_DIR)

## Decay parameter

`DECAY_RATE` is the fraction of weight a season keeps for each year further back from the anchor season (2025-26). At 0.75: last season (2024-25) counts at 75%, two seasons back at 56%, ..., 2015-16 (10 years back) at about 5.6% -- present but nearly negligible, which is the point of decay over a hard cutoff.

This used to be picked by feel (0.6 / 0.75 / 0.85 were all floated as reasonable-sounding options, with 0.75 shipped as a placeholder). It's now validated the same honest way the box-score prior's half-life was tuned in `souped_up_rapm.ipynb`: for several anchor years, build the decayed multi-year RAPM through that year at each candidate rate and check how well it predicts the *following* season's real single-year RAPM. See the validation cell right below -- it sets `DECAY_RATE` from that result rather than the hand-picked value.

In [ ]:
ANCHOR_SEASON = 2025  # 2025-26, start-year label matching this project's convention throughout
# DECAY_RATE is set by the validation cell below, not hardcoded here.

SEASON_FILES = {
    2015: "rapm_possessions2015.csv",
    2016: "rapm_possessions2016.csv",
    2017: "rapm_possessions2017.csv",
    2018: "rapm_possessions2018.csv",
    2019: "rapm_possessions2019.csv",
    2020: "rapm_possessions2020.csv",
    2021: "rapm_possessions2021.csv",
    2022: "rapm_possessions2022.csv",
    2023: "rapm_possessions2023.csv",
    2024: "rapm_possessions2024.csv",
    2025: "rapm_possessions2025.csv",
}


### Validating DECAY_RATE against held-out data

Same fold structure as the k-tuning in `souped_up_rapm.ipynb`: 5 anchor years (2019, 2021, 2022, 2023, 2024), each treated as a fold -- build the decayed RAPM through that anchor year at a given decay rate, then correlate it with the *next* season's real single-year RAPM. Report the mean correlation and its standard error across the 5 folds (not just the pooled correlation), so a genuine peak can be told apart from noise the way the k-tuning curve in metric #4 could not.

This refits RAPM ~65 times (13 candidate rates x 5 anchors), so it takes several minutes to run -- much heavier than the other tuning cells in this project, but it's a one-time validation, not something that needs to run every time.

In [ ]:
DECAY_GRID = [0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
VALIDATION_ANCHORS = [2019, 2021, 2022, 2023, 2024]

# target: single-season plain RAPM for the season right after each anchor
_target_years = sorted(set(a + 1 for a in VALIDATION_ANCHORS))
_season_fits = {}
for year in _target_years:
    poss = pd.read_csv(os.path.join(POSS_DIR, f"rapm_possessions{year}.csv"))
    poss = poss[poss["possessions"] > 0].copy()
    players = rapm.build_player_list(poss)
    Xd = rapm.build_design_matrix(poss, players)
    poss["PPP"] = 100 * poss["points"] / poss["possessions"]
    res, _ = rapm.calculate_rapm(Xd, poss["PPP"].to_numpy(), poss["possessions"].to_numpy(), rapm.DEFAULT_LAMBDAS, "RAPM", players)
    _season_fits[year] = res.set_index("playerId")["RAPM"]

# cache the decay-rate-independent pieces (pooled possessions + design matrix) once per anchor
_anchor_cache = {}
for anchor in VALIDATION_ANCHORS:
    frames = []
    for season in range(min(SEASON_FILES), anchor + 1):
        df = pd.read_csv(os.path.join(POSS_DIR, SEASON_FILES[season]))
        df = df[df["possessions"] > 0].copy()
        df["years_ago"] = anchor - season
        frames.append(df)
    pooled = pd.concat(frames, ignore_index=True)
    players = rapm.build_player_list(pooled)
    Xd = rapm.build_design_matrix(pooled, players)
    pooled["PPP"] = 100 * pooled["points"] / pooled["possessions"]
    _anchor_cache[anchor] = {
        "poss": pooled["possessions"].to_numpy(), "years_ago": pooled["years_ago"].to_numpy(),
        "PPP": pooled["PPP"].to_numpy(), "Xd": Xd, "players": players,
    }
    print(f"anchor cache built: {anchor} ({len(pooled)} rows, {len(players)} players)")

_rows = []
for rate in DECAY_GRID:
    for anchor in VALIDATION_ANCHORS:
        c = _anchor_cache[anchor]
        w = c["poss"] * (rate ** c["years_ago"])
        res, _ = rapm.calculate_rapm(c["Xd"], c["PPP"], w, rapm.DEFAULT_LAMBDAS, "RAPM_decay_test", c["players"])
        res = res.set_index("playerId")
        target = _season_fits[anchor + 1]
        common = res.index.intersection(target.index)
        corr = res["RAPM_decay_test"].reindex(common).corr(target.reindex(common))
        _rows.append({"anchor": anchor, "decay_rate": rate, "corr": corr})
    print(f"rate={rate} done")

decay_validation = pd.DataFrame(_rows)
decay_summary = decay_validation.groupby("decay_rate")["corr"].agg(["mean", "std", "count"]).reset_index()
decay_summary["se"] = decay_summary["std"] / np.sqrt(decay_summary["count"])
decay_summary = decay_summary.rename(columns={"mean": "corr"})
print()
print(decay_summary.to_string(index=False))

# The 1-SE rule is the right tool when a curve is flat and hits a wall right at the edge of the
# tested grid -- that pattern means the true optimum might sit even further out, and the rule protects
# against chasing sampling noise out to that edge (this is exactly what happened with k elsewhere in
# this project, and with this project's own box half-life curve). It is the WRONG tool here: the table
# above is a clean, symmetric interior peak. Correlation rises from 0.398 (rate=0.3) up through 0.440
# at rate=0.75, then falls back down through 0.421 at rate=1.0 (no decay at all). Both edges of the
# tested range score clearly worse than the peak, so there's no boundary-artifact risk to guard
# against, and pushing to the far edge of the SE band (rate=0.95, a 13.5-year half-life) doesn't
# correct for anything real -- it just trades a well-supported point estimate for a much less
# plausible one. A 13.5-year half-life implies a possession from a player's rookie year is still
# ~90% as informative as one from last week, which doesn't survive basic scrutiny: players change
# role, team context, and physical peak substantially over a career. Unlike k (where over-shrinking
# toward an unproven prior is a specific, known failure mode that justifies erring away from it),
# there's no equivalent domain reason to bias decay rate toward "keep more old history" by default.
# So: just use the argmax. It's both the best point estimate AND the one that passes a face-validity
# check, which is the ordinary case for a real interior peak.
best_row = decay_summary.loc[decay_summary["corr"].idxmax()]
DECAY_RATE = float(best_row["decay_rate"])
half_life_years = np.log(0.5) / np.log(DECAY_RATE)
print(f"Argmax rate: {DECAY_RATE} (corr={best_row['corr']:.4f}, se={best_row['se']:.4f})")
print(f"Chosen DECAY_RATE (argmax -- a genuine interior peak, not a boundary artifact): {DECAY_RATE}")
print(f"Half-life: {half_life_years:.2f} years")


In [ ]:
print("season  years_ago  weight  (DECAY_RATE = {:.2f}, validated above)".format(DECAY_RATE))
for season in sorted(SEASON_FILES):
    years_ago = ANCHOR_SEASON - season
    weight = DECAY_RATE ** years_ago
    print(f"{season}    {years_ago:>9}  {weight:.4f}")


In [ ]:
frames = []
for season, fname in SEASON_FILES.items():
    path = os.path.join(POSS_DIR, fname)
    df = pd.read_csv(path)
    years_ago = ANCHOR_SEASON - season
    df["season"] = season
    df["recency_weight"] = DECAY_RATE ** years_ago
    frames.append(df)
    print(f"{fname}: {len(df)} possession rows, season={season}, recency_weight={DECAY_RATE ** years_ago:.4f}")

possessions_decayed = pd.concat(frames, ignore_index=True)
print(f"\nTotal: {len(possessions_decayed)} possession rows across {len(SEASON_FILES)} seasons")

## Player names

Same approach as `run_rapm.ipynb`: `CommonAllPlayers` scoped to the anchor season so names resolve for anyone who played recently. Older players who only appear in older seasons and not 2025-26 will fall out of the name merge (an inner join, matching the existing pipeline's convention) -- acceptable here since the whole point of recency weighting is to describe *current* value, not resurrect retired players' possessions.

In [ ]:
player_names_raw = nba_client.get_all_players(f"{ANCHOR_SEASON}-{str(ANCHOR_SEASON+1)[2:]}")
player_names = player_names_raw.rename(columns={"PERSON_ID": "person_id", "DISPLAY_FIRST_LAST": "player"})
player_names[["person_id", "player"]].head()

## Run the decayed RAPM

In [ ]:
results_decayed, intercept_decayed = rapm.run(
    possessions_decayed, name="RAPM_decayed", player_names=player_names, extra_weight_col="recency_weight"
)
print("intercept:", intercept_decayed)

out_path = os.path.join(OUT_DIR, f"rapm_decayed_{ANCHOR_SEASON}.csv")
results_decayed.to_csv(out_path, index=False)
print(f"Wrote {out_path}")
results_decayed.head(20)

## Sanity checks

1. Does RidgeCV land on a boundary lambda (0.01 or 0.1)? Same thing to watch for as the existing pipeline -- if so, widen `DEFAULT_LAMBDAS` in `rapm_pipeline/rapm.py`.
2. Compare against the equal-pooled 5-year RAPM you already have -- decay-weighting should agree closely with 5yr for players whose recent play looks like their whole career, and should diverge specifically for players who've clearly changed (a young player who broke out, an older player who's declined, someone who changed role or team). Those divergences are the actual finding this metric is for.

In [ ]:
existing_5yr_path = os.path.join(POSS_DIR, "..", "data", "rapm_5yr_2025_26.csv")
existing_5yr_path = os.path.join(POSS_DIR, "rapm_5yr_2025_26.csv")
if not os.path.exists(existing_5yr_path):
    # fall back to wherever it actually lives if the above guess is wrong
    existing_5yr_path = os.path.join(RAPM_PIPELINE_DIR, "data", "rapm_5yr_2025_26.csv")

five_yr = pd.read_csv(existing_5yr_path)
cmp = results_decayed.merge(five_yr, on="playerId", suffixes=("", "_5yr"))
print(f"corr(RAPM_decayed, RAPM_5yr): {cmp['RAPM_decayed'].corr(cmp['RAPM_5yr']):.4f}")

cmp["abs_diff"] = (cmp["RAPM_decayed"] - cmp["RAPM_5yr"]).abs()
print("\nBiggest divergences from the equal-pooled 5-year RAPM (players whose recent play looks different from their 5-year average):")
name_col = "player" if "player" in cmp.columns else "playerId"
cmp.sort_values("abs_diff", ascending=False).head(15)[[name_col, "RAPM_decayed", "RAPM_5yr", "abs_diff"]]

## Alpha-calibration bug found and fixed while building this notebook

The first version of this notebook ran cleanly but produced a suspicious pattern: current stars (Jokić, SGA, Wembanyama, Giannis, Chet Holmgren) came out *lower* in decayed RAPM than in the equal-pooled 5yr RAPM, while replacement-level players came out higher -- a textbook "compression toward the mean" symptom, not a real "these guys got worse" finding.

Root cause: `rapm_pipeline/rapm.py`'s `lambda_to_alpha` calibrates the ridge penalty off the raw *row count* of the design matrix. In every call site up to now that's harmless, because each row is exactly one possession with weight 1 -- row count and total sample weight are identical. Recency weighting breaks that equivalence: the 11-year decayed panel has ~2.17x more raw rows than the 5-year equal-pooled panel, but because old seasons get decayed down toward ~0 weight, it only has ~0.76x the *actual* (weighted) information content. Calibrating alpha off the inflated row count made the decayed fit far more regularized than its true sample size justified, shrinking everyone's estimate toward zero -- hardest on players who were already far from zero (stars in both directions).

**Fix (in `rapm_pipeline/rapm.py`, `calculate_rapm`):** calibrate alpha off `possession_weights.sum()` (the total weighted sample) instead of `design_matrix.shape[0]` (the raw row count). This is a no-op for every existing call site -- unweighted possessions always have `sum(weights) == shape[0]` exactly, confirmed for the 5yr panel (both equal 1,160,770) -- so single-season, 3yr, and 5yr equal-pooled RAPM are byte-for-byte unaffected. It only changes behavior when `extra_weight_col` is used, which is exactly the case that needed fixing.

After the fix, decayed vs 5yr correlation rose from 0.9570 to 0.9760, and the pathological pattern disappeared: Jokić -0.93 vs 5yr (was -3.61), SGA -0.49 (was -2.85), Wembanyama **+0.50** (was -2.60), Giannis +0.01 (was -2.50), Kawhi +0.51, Chet Holmgren +0.16 -- i.e. the players whose recent single-year form is at or above their career level now move in the *right* direction relative to equal-pooling, not the wrong one. The cells below build in a direct check of that (single-year RAPM trend per player) rather than just trusting the aggregate correlation.

## Cross-check against single-year RAPM

The most direct sanity check for a recency-weighted metric: pull single-year RAPM for each season a player was active and see whether their trend is actually up or down. Decayed RAPM should sit closer to a player's *recent* single-year values than the 5yr equal-pooled version does, and should never move in the opposite direction from an obvious single-year trend (that's what the alpha-calibration bug above produced, and why this check exists).

In [ ]:
# Single-year RAPM for every season 2015-2025 -- cheap (~1-2s per season fit),
# so just compute all of them rather than special-casing a handful of years.
single_year_frames = []
for season, fname in SEASON_FILES.items():
    df = pd.read_csv(os.path.join(POSS_DIR, fname))
    res, _ = rapm.run(df, name="RAPM")
    res["Season"] = season
    single_year_frames.append(res[["playerId", "Season", "RAPM"]])

single_year_rapm = pd.concat(single_year_frames, ignore_index=True)
single_year_rapm = single_year_rapm.merge(player_names[["person_id", "player"]], left_on="playerId", right_on="person_id", how="left")
single_year_rapm.to_csv(os.path.join(OUT_DIR, "single_year_rapm_2015_2025.csv"), index=False)
single_year_rapm.head()

In [ ]:
# Spot-check: for a handful of current stars, lay the single-year trend
# next to decayed RAPM and 5yr equal-pooled RAPM. The decayed value should
# read as plausible given the single-year trajectory -- never a decline for
# someone whose single-year RAPM has been climbing every year.
SPOT_CHECK_IDS = {
    203999: "Nikola Jokic",
    1628983: "Shai Gilgeous-Alexander",
    1641705: "Victor Wembanyama",
    203507: "Giannis Antetokounmpo",
    202695: "Kawhi Leonard",
    1631096: "Chet Holmgren",
}

for pid, label in SPOT_CHECK_IDS.items():
    trend = single_year_rapm[single_year_rapm["playerId"] == pid].sort_values("Season")
    dec_row = results_decayed[results_decayed["playerId"] == pid]
    fiveyr_row = five_yr[five_yr["playerId"] == pid]
    dec_val = dec_row["RAPM_decayed"].iloc[0] if len(dec_row) else float("nan")
    fiveyr_val = fiveyr_row["RAPM_5yr"].iloc[0] if len(fiveyr_row) else float("nan")
    print(f"=== {label} ===")
    print(trend[["Season", "RAPM"]].to_string(index=False))
    print(f"decayed={dec_val:.2f}  5yr_equal_pooled={fiveyr_val:.2f}\n")